# Payment Slip & Work Certificate Anomaly Detection — OCR → Text → Embedding → Score

This notebook classifies **payment slips** and **work certificates** as **normal** or **anomaly** by
comparing each one against a set of **known-genuine** documents. It is fully self-contained:
everything needed to run it on a fresh machine — requirements, folder layout, and the complete
pipeline code — is inside this file. Nothing else from the project is required.

### Why text, not image
Bank statements have a stable, bank-issued layout, so the sibling notebook scores them by their
*appearance*. Payment slips and work certificates differ from company to company, so there is no
"normal layout" to learn — instead we read the **text** out of each document (OCR) and score its
**meaning**.

### The method in one paragraph
We OCR each document into text, turn that text into a 768-number "fingerprint" (an *embedding*) with a
frozen multilingual model (**multilingual-E5**), model the fingerprints of genuine documents as one
cloud of points (a Gaussian), and measure how far each new document falls from the centre of that
cloud — its **Mahalanobis distance**. Documents far outside the genuine cloud are flagged. This is a
*one-class* approach (the PaDiM paradigm): we only ever learn what **genuine** looks like — no
forged/fake examples are needed, and the alarm threshold comes purely from how much genuine documents
vary among themselves.

### Two independent signals decide the verdict
Alongside the embedding we apply a **required-field checklist**: a genuine document must contain
certain fields, matched by keyword **or by meaning** (so "Received Income" counts as take-home pay).
A document is flagged **anomaly** if **either**:
- its embedding score exceeds the alarm threshold (its text is unusual versus genuine), **or**
- it is **missing a required field** (the checklist fails).

**Required fields**
- **Payment slip:** worker name **and** company name **and** period **and** income (income = take-home pay **or** gross income **or** basic salary).
- **Work certificate:** worker name **and** company name **and** role / job grade.

### Per document (not per page)
Unlike the image notebook (which scores every page), the text of all pages is merged and the document
is scored as a whole — a payslip's fields can appear on any page.

### The four stages
| Stage | Input → Output | Section |
|-------|----------------|---------|
| 1. Extract | PDF/image bytes → merged OCR text (all pages) | Run · Stage 1 |
| 2. Embed | text → 768-dim vector (frozen E5, on CPU) | Run · Stage 2 |
| 3. Fit "normal" model | genuine vectors → Gaussian + alarm threshold | Run · Stage 3 |
| 4. Score + rules | vector → distance; text → field checklist; **verdict = distance OR missing-field** | Run · Stage 4 |

### How to run it (any machine)
1. Install the **requirements** (next section) — all pure `pip`.
2. Put your documents in the **folder structure** shown two sections down, and set `DOC_TYPE`.
3. From the menu choose **Run → Run All Cells**. The first run downloads the E5 model weights
   (~1.1 GB) and the RapidOCR models (~15 MB) once, then caches them. On a laptop CPU, expect a few
   seconds per document.

> **Privacy note — please read.** OCR text contains personal data (names, salaries). This notebook
> shows a short **extraction sample** so you can see what was read; it is designed to run
> **locally / on-prem**. But a saved `.ipynb` stores its cell outputs, so **Clear All Outputs before
> sharing or committing this file**. The results **CSV is non-identifying** (scores + field names
> only) and is safe to keep.

## 1 · Requirements

**Python:** 3.10 or newer. **Everything installs with `pip` — there is no separate system software to
install.** (PDF reading uses `pypdfium2`, which bundles Google's PDFium engine inside the pip package;
OCR uses `rapidocr-onnxruntime`, which bundles its ONNX models — so, unlike a Tesseract/Poppler route,
nothing extra is needed on the machine.)

| Package | Purpose |
|---------|---------|
| `torch` | runs the E5 neural network |
| `transformers` | loads the E5 model + tokenizer from Hugging Face |
| `rapidocr-onnxruntime` | OCR — reads text out of scans/photos (Indonesian/Latin), no language pack |
| `pypdfium2` | renders PDF pages to images and reads the embedded text layer (self-contained; **no Poppler**) |
| `pillow` | image decoding/handling |
| `numpy` | the maths (Gaussian, Mahalanobis distance, cosine similarity) |
| `matplotlib` | the charts |

Run the cell below once (safe to re-run). That's the whole setup.

> **Offline note:** the first run downloads the E5 weights from Hugging Face (~1.1 GB, cached
> afterwards) and the RapidOCR ONNX models (~15 MB). If the target machine has no internet, copy the
> sender's `~/.cache/huggingface` folder (and the cached RapidOCR models) to the same path on the
> target machine, or set `HF_HOME` to a folder that already contains the weights.

In [ ]:
# Install everything. Pure pip — no system software needed. Safe to re-run (pip skips what's present).
%pip install -q "torch>=2.2" "transformers>=4.40" "rapidocr-onnxruntime>=1.3" "pypdfium2>=4" "pillow>=10" "numpy>=1.26" "matplotlib>=3.8"

In [ ]:
# Environment check — confirms every requirement is present. There is NO system dependency to set up:
# PDF reading (pypdfium2) and OCR (rapidocr-onnxruntime) both ship their engines inside the pip package.
import importlib.util, sys

print("Python:", sys.version.split()[0], "(need >= 3.10)\n")
print("Python packages:")
all_ok = True
for mod, pretty in [("torch","torch"),("transformers","transformers"),
                    ("rapidocr_onnxruntime","rapidocr-onnxruntime"),("pypdfium2","pypdfium2"),
                    ("PIL","pillow"),("numpy","numpy"),("matplotlib","matplotlib")]:
    ok = importlib.util.find_spec(mod) is not None
    all_ok = all_ok and ok
    print(f"  [{'OK ' if ok else 'MISSING'}] {pretty}")

print("\n" + ("All requirements present — no system software needed."
              if all_ok else
              "Something is MISSING — run the install cell above, then re-run this cell."))

## 2 · Folder structure

Keep this notebook at the **root of a working folder**, with `reference/` and `data/` beside it. The
`result/` folder is created for you.

```
your_working_folder/
├── text_based_anomaly_detection.ipynb    ← this notebook
│
├── reference/
│   ├── payment_slip/         ← KNOWN-GENUINE payslips. These define "normal".
│   │   ├── AAZGG53.pdf           Use at least 3 (the model needs 3+); 12–20 is much better.
│   │   └── …                     One clean, genuine document per layout you expect.
│   └── work_certificate/     ← KNOWN-GENUINE certificates.
│       └── …
│
├── data/
│   ├── payment_slip/         ← the payslips you want to CHECK (classify).
│   │   └── …                     Scored against the genuine set above.
│   └── work_certificate/     ← the certificates you want to check.
│       └── …
│
└── result/                   ← created automatically; holds the timestamped results CSV.
    └── payment_slip/
        └── 20260727_101500_results.csv
```

| Path | Role |
|------|------|
| `reference/<type>/` | The ground truth of "genuine". The model is built **only** from these, so they must all be real, clean documents. More here → more reliable threshold. |
| `data/<type>/` | The documents under question. Each is labelled `normal` or `anomaly`. |
| `result/<type>/` | Output. One `…_results.csv` per run (timestamped, so runs never overwrite). |
| accepted file types | `.pdf`, `.jpg`, `.jpeg`, `.png`, `.tif`, `.tiff`. All pages of a PDF are read. Sub-folders/other types are ignored. |

Pick which document type to analyse with **`DOC_TYPE`** in the next cell (`"payment_slip"` or
`"work_certificate"`); both rule sets are built in. To analyse the other type, change `DOC_TYPE` and
Run-All again.

**No ground-truth reference? Use *unsupervised mode*.** If you just have a pile of documents and no
separate set of known-genuine ones, put them all in `reference/<type>/`, leave `data/<type>/` empty,
and set `UNSUPERVISED = True` in the config cell. The notebook then scores every document by how
unusual it is *versus the rest of the pile* (leave-one-out) and flags the most unusual `PERCENTILE`% —
no ground truth and no manual threshold needed. The **required-field checklist still applies**, since
it needs no reference set. See §6 at the bottom.

> The next cell **auto-detects** the working folder by looking for a `reference/` folder in the
> notebook's directory and its parents. To force a location, hard-code `BASE_DIR`.

In [ ]:
from pathlib import Path
import torch

# ── Configuration you can edit ──────────────────────────────────────────────
DOC_TYPE   = "payment_slip"                     # "payment_slip" or "work_certificate"
MODEL      = "intfloat/multilingual-e5-base"    # frozen text-embedding model (E5, 768-dim, multilingual)
PERCENTILE = 95.0                               # alarm threshold = this percentile of genuine scores

# DEVICE: auto-detects a GPU (CUDA) and falls back to CPU if none is available/usable, so this
# notebook runs unchanged on a GPU or CPU-only machine. To force one or the other, hard-code
# DEVICE = "cuda" or DEVICE = "cpu" instead of the line below.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# UNSUPERVISED = True  ->  you have NO ground-truth reference set. Put your whole pile of documents in
# reference/<type>/ (leave data/<type>/ empty). Every document is then scored by how unusual it is
# versus the REST of the same set (leave-one-out), and the most unusual PERCENTILE% are flagged.
# The required-field checklist below still applies — it needs no reference set.
UNSUPERVISED = False

# Required-field rules per document type. A requirement is satisfied if ANY of its anchor phrases is
# found — by exact/synonym match OR by E5 semantic similarity >= semantic_threshold (so a paraphrase
# like "Received Income" satisfies `income`). A document must satisfy EVERY requirement to pass.
RULES = {
    "payment_slip": {
        "semantic_threshold": 0.83,
        "requirements": {
            "worker_name":  ["worker name", "employee name", "nama", "nama karyawan", "nama pegawai", "nama lengkap"],
            "company_name": ["company name", "company", "nama perusahaan", "perusahaan", "PT", "CV", "instansi", "employer"],
            "period":       ["pay period", "period", "periode", "periode gaji", "bulan gaji", "bulan", "masa gaji", "payroll period"],
            "income":       ["take home pay", "received income", "net income", "net pay", "net salary",
                             "gaji bersih", "penghasilan bersih", "pendapatan bersih", "gaji diterima", "penghasilan diterima",
                             "gross income", "gross salary", "penghasilan bruto", "gaji kotor", "total pendapatan", "total penghasilan",
                             "basic salary", "base salary", "gaji pokok", "upah pokok"],
        },
    },
    "work_certificate": {
        "semantic_threshold": 0.84,
        "requirements": {
            "worker_name":  ["worker name", "employee name", "nama", "nama karyawan", "nama pegawai", "nama lengkap"],
            "company_name": ["company name", "company", "nama perusahaan", "perusahaan", "PT", "CV", "instansi", "kantor", "employer"],
            "role":         ["job grade", "job title", "position", "role", "jabatan", "posisi", "golongan", "pangkat", "jabatan sebagai"],
        },
    },
}
# ────────────────────────────────────────────────────────────────────────────
assert DOC_TYPE in RULES, f"DOC_TYPE must be one of {list(RULES)}"
SPEC = RULES[DOC_TYPE]

def _find_base(start: Path, doc_type: str) -> Path:
    for cand in (start, *start.parents):
        if (cand / "reference" / doc_type).is_dir() or (cand / "reference").is_dir():
            return cand
    return start
BASE_DIR = _find_base(Path.cwd(), DOC_TYPE)

REFERENCE_DIR = BASE_DIR / "reference" / DOC_TYPE
DATA_DIR      = BASE_DIR / "data" / DOC_TYPE
RESULT_DIR    = BASE_DIR / "result" / DOC_TYPE
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print(f"DEVICE   : {DEVICE}" + (f"  ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else "  (no GPU detected — set up CUDA drivers/torch to use one)"))
print("DOC_TYPE :", DOC_TYPE, " | UNSUPERVISED :", UNSUPERVISED)
print("required :", list(SPEC["requirements"]), " | semantic_threshold =", SPEC["semantic_threshold"])
print("BASE_DIR :", BASE_DIR)
for label, d in [("reference", REFERENCE_DIR), ("data", DATA_DIR)]:
    if label == "data" and UNSUPERVISED:
        print(f"  {label:10}: {d}   [not used — UNSUPERVISED mode]")
        continue
    status = "OK" if d.is_dir() else "MISSING — create it and add documents"
    print(f"  {label:10}: {d}   [{status}]")

## 3 · Pipeline code (self-contained)

The next five cells contain the entire pipeline. They mirror the production service so results match
it closely, but are inlined here so this notebook needs nothing else. You don't need to change anything.

- **3.1 `document_to_text`** — PDF/image bytes → merged OCR text, all pages *(mirrors `pipeline/text_extract.py` + `ocr.py`)*
- **3.2 `E5TextEmbedder`** — text → 768-dim vector *(mirrors `pipeline/embedding.py`)*
- **3.3 `NormalModel`** — fit the genuine cloud, derive the threshold, score *(mirrors `anomaly.py`)*
- **3.4 required-field rules** — exact + semantic field matching *(mirrors `rules/content_checks.py` + `semantic_match.py`)*
- **3.5 plotting helpers** — the score distribution and the required-field grid

In [ ]:
# 3.1 — Extract: raw document bytes -> merged plain text. Every page of a PDF is rendered (pypdfium2,
#       no system dependency) and OCR'd (RapidOCR), and the embedded PDF text layer is merged in.
#       A plain image (jpg/png/tif) is OCR'd directly. Text is held in memory only.
import io, re
import numpy as np
from PIL import Image

PDF_RENDER_DPI = 200   # resolution used to rasterize PDF pages before OCR (200 is a good default)

class RapidOCREngine:
    """RapidOCR (ONNX PP-OCR). Reads Latin-script text (Indonesian included) with no language pack.
    Models download once (~15 MB) and cache. Build ONE engine and reuse it across documents."""
    def __init__(self):
        from rapidocr_onnxruntime import RapidOCR
        self._engine = RapidOCR()
    def text(self, image) -> str:
        result, _ = self._engine(np.asarray(image.convert("RGB")))
        return "\n".join(row[1] for row in result) if result else ""

def _pdf_pages_and_layer(data: bytes):
    """Return (list of RGB page images, embedded text-layer string) for a PDF."""
    import pypdfium2 as pdfium
    pdf = pdfium.PdfDocument(data)
    images, layer = [], []
    try:
        if len(pdf) == 0:
            raise ValueError("PDF contains no pages")
        for i in range(len(pdf)):
            page = pdf[i]
            images.append(page.render(scale=PDF_RENDER_DPI / 72).to_pil().convert("RGB"))
            tp = page.get_textpage()
            layer.append(tp.get_text_bounded())     # embedded text layer (empty for scanned PDFs)
            tp.close(); page.close()
        return images, "\n".join(layer)
    finally:
        pdf.close()

_NO_LETTERS = re.compile(r"^[^A-Za-z]+$")
def _merge_dedupe(parts) -> str:
    """Join fragments, dropping blank and exact-duplicate lines (the 'Halaman N' chrome repeats)."""
    seen, out = set(), []
    for part in parts:
        for line in part.splitlines():
            s = line.strip()
            if not s or s in seen:
                continue
            seen.add(s); out.append(s)
    return "\n".join(out)

def document_to_text(data: bytes, ocr: RapidOCREngine) -> str:
    if not data:
        raise ValueError("empty document")
    parts = []
    if data[:5] == b"%PDF-":
        images, layer = _pdf_pages_and_layer(data)
        if layer.strip():
            parts.append(layer)
        parts.extend(ocr.text(img) for img in images)   # OCR every page (photos have a junk layer)
    else:
        parts.append(ocr.text(Image.open(io.BytesIO(data)).convert("RGB")))
    return _merge_dedupe(parts)

In [ ]:
# 3.2 — Embed: text -> a fixed 768-dim vector, using a FROZEN multilingual-E5 model (no training).
#       Readout = mean of the token embeddings (attention-masked), then L2-normalised — the standard
#       E5 recipe. E5 is asymmetric: documents get a "passage: " prefix, field concepts a "query: ".
class E5TextEmbedder:
    def __init__(self, model_id="intfloat/multilingual-e5-base", device="cpu"):
        import torch
        from transformers import AutoModel, AutoTokenizer
        self.model_id, self._device, self._torch = model_id, device, torch
        self._tok = AutoTokenizer.from_pretrained(model_id)
        # use_safetensors=True: this pinned torch (2.2.2, needed for CPU compatibility - see
        # requirements.txt) is older than what recent transformers demands for the legacy
        # torch.load pickle format (CVE-2025-32434). Safetensors loading is unaffected by that
        # restriction, so we force it explicitly rather than upgrading torch (which would crash
        # on this CPU - see requirements.txt for why).
        self._model = AutoModel.from_pretrained(model_id, use_safetensors=True).to(device).eval()
        self.dim = int(self._model.config.hidden_size)

    def embed_text(self, text, prefix="passage: ") -> tuple:
        torch = self._torch
        inputs = self._tok(prefix + (text or " "), return_tensors="pt",
                           truncation=True, max_length=512, padding=True).to(self._device)
        with torch.no_grad():
            hidden = self._model(**inputs).last_hidden_state          # (1, seq, dim)
        mask = inputs["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        vec = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        vec = vec[0]
        vec = vec / vec.norm(p=2).clamp(min=1e-12)                    # unit-normalise (cosine space)
        return tuple(vec.detach().cpu().numpy().astype("float32").tolist())

In [ ]:
# 3.3 — The "normal" model: fit a Gaussian to the genuine vectors and score new ones by
#       Mahalanobis distance. Pure numpy. Mirrors anomaly.py exactly.
from dataclasses import dataclass

def _shrunk_covariance(centered):
    # Ledoit-Wolf shrinkage toward a scaled identity, so the covariance is invertible even with
    # far fewer samples (e.g. 15) than dimensions (768).
    n, d = centered.shape
    sample = (centered.T @ centered) / n
    mean_var = np.trace(sample) / d
    denom = ((sample - mean_var * np.eye(d)) ** 2).sum() / d
    numer = sum(((np.outer(centered[k], centered[k]) - sample) ** 2).sum() for k in range(n))
    beta = min(numer / (n**2 * d), denom)
    shrink = beta / denom if denom > 0 else 1.0
    return shrink * mean_var * np.eye(d) + (1 - shrink) * sample, float(shrink)

def _fit_gaussian(X):
    mu = X.mean(axis=0)
    cov, shrink = _shrunk_covariance(X - mu)
    return mu, np.linalg.inv(cov), shrink

def _mahalanobis(x, mu, cov_inv):
    v = x - mu
    return float(np.sqrt(max(v @ cov_inv @ v, 0.0)))

@dataclass
class NormalModel:
    mu: np.ndarray
    cov_inv: np.ndarray
    threshold: float
    shrinkage: float
    ref_scores: np.ndarray     # leave-one-out Mahalanobis scores of the reference set
    pca_basis: np.ndarray      # (2, d) top-2 principal directions of the reference set
    ref_pca: np.ndarray        # (n, 2) reference set projected to 2-D
    explained_var: tuple

    @classmethod
    def fit(cls, X, threshold_percentile: float = 95.0) -> "NormalModel":
        X = np.asarray(X, dtype=float)
        n = len(X)
        if n < 3:
            raise ValueError(f"need at least 3 reference vectors to fit a model, got {n}")
        mu, cov_inv, shrink = _fit_gaussian(X)
        # Honest threshold: score each reference vector against the OTHERS (leave-one-out).
        loo = np.array([_mahalanobis(X[i], *_fit_gaussian(np.delete(X, i, axis=0))[:2]) for i in range(n)])
        threshold = float(np.percentile(loo, threshold_percentile))
        centered = X - mu
        _, singular, vt = np.linalg.svd(centered, full_matrices=False)
        variance = (singular**2) / (singular**2).sum()
        basis = vt[:2]
        return cls(mu, cov_inv, threshold, shrink, loo, basis, centered @ basis.T,
                   (float(variance[0]), float(variance[1])))

    def score(self, x) -> float:
        return _mahalanobis(np.asarray(x, dtype=float), self.mu, self.cov_inv)

In [ ]:
# 3.4 — Required-field rules. A requirement is satisfied if any anchor matches by exact/synonym
#       match (word-boundary, accent/case-insensitive) OR by E5 semantic similarity >= threshold.
#       Only field NAMES + booleans + similarities leave this code — never the matched text.
import unicodedata

def _normalize(text: str) -> str:
    d = unicodedata.normalize("NFKD", text)
    return "".join(c for c in d if not unicodedata.combining(c)).casefold()

def _synonym_pattern(s: str):
    toks = [re.escape(t) for t in _normalize(s).split()]
    return re.compile(r"\b" + r"\s+".join(toks) + r"\b")

def _segments(text: str):
    return [ln.strip() for ln in text.splitlines() if ln.strip() and not _NO_LETTERS.match(ln.strip())]

class SemanticMatcher:
    """E5 similarity between a requirement's anchor concepts and a document's lines."""
    def __init__(self, embedder):
        self._emb = embedder
        self._cache = {}
    def _anchor_vec(self, phrase):
        v = self._cache.get(phrase)
        if v is None:
            v = np.asarray(self._emb.embed_text(phrase, prefix="query: "), dtype=float)
            self._cache[phrase] = v
        return v
    def line_embeddings(self, text):
        segs = _segments(text)
        if not segs:
            return np.zeros((0, 0))
        return np.array([self._emb.embed_text(s, prefix="passage: ") for s in segs], dtype=float)
    def max_similarity(self, anchors, line_embeddings):
        if line_embeddings.size == 0 or not anchors:
            return 0.0
        A = np.array([self._anchor_vec(a) for a in anchors], dtype=float)
        return float((line_embeddings @ A.T).max())   # cosine (vectors are unit-normalised)

def evaluate_content(text, spec, matcher):
    """Return (content_ok, missing, detail). detail[req] = (method, similarity):
    method is 'exact', 'semantic', or 'missing'."""
    norm = _normalize(text)
    detail, satisfied, unmatched = {}, set(), []
    for req, anchors in spec["requirements"].items():
        if any(_synonym_pattern(a).search(norm) for a in anchors):
            detail[req] = ("exact", 1.0); satisfied.add(req)
        else:
            unmatched.append((req, anchors))
    if unmatched:
        line_embs = matcher.line_embeddings(text)
        for req, anchors in unmatched:
            sim = matcher.max_similarity(anchors, line_embs)
            if sim >= spec["semantic_threshold"]:
                detail[req] = ("semantic", sim); satisfied.add(req)
            else:
                detail[req] = ("missing", sim)
    reqs = list(spec["requirements"])
    missing = tuple(r for r in reqs if r not in satisfied)
    return (len(missing) == 0), missing, detail

In [ ]:
# 3.5 — Plotting helpers. Every chart labels its X and Y axes. (They save a PNG and display it.)
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

TEAL, RED, GREY, INK = "#0c7a71", "#b3352b", "#7b8783", "#16201f"
SUPPORTED = {".pdf", ".jpg", ".jpeg", ".png", ".tif", ".tiff"}

def list_documents(folder: Path):
    if not folder.is_dir():
        raise SystemExit(f"folder not found: {folder}")
    return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED)

def _short_name(name: str, limit: int = 30) -> str:
    if len(name) <= limit:
        return name
    head = (limit - 1) * 3 // 5
    return f"{name[:head]}…{name[-(limit - 1 - head):]}"

def plot_distribution(model, names, scores, verdicts, content_flagged, path, title_label="",
                      unsupervised=False):
    """One dot per DOCUMENT.
    Supervised: teal = genuine reference cloud (leave-one-out) and normal data; red = anomaly.
    Unsupervised: a single population — each document's LEAVE-ONE-OUT score — coloured by verdict,
    and only anomalies are labelled so a pile of hundreds stays readable.
    In both modes a hollow red ring marks a doc flagged for a MISSING REQUIRED FIELD (its embedding
    score alone may be below the threshold), so a red mark left of the line is explained."""
    fig, ax = plt.subplots(figsize=(10, 4.8))
    hi = max([model.threshold, float(model.ref_scores.max()), *scores]) * 1.12
    ref_y, data_y = 0.14, 0.30
    levels = [0.44, 0.56, 0.68, 0.80, 0.92]
    ax.axvspan(model.threshold, hi, color=RED, alpha=0.06, lw=0)
    ax.axvline(model.threshold, color=GREY, ls="--", lw=1.3)
    ax.text(model.threshold + hi * 0.008, 0.99, f"threshold {model.threshold:.0f}", color=GREY,
            fontsize=8, ha="left", va="top", transform=ax.get_xaxis_transform())
    rng = np.random.default_rng(0)
    if not unsupervised:   # supervised: show the genuine reference cloud as its own lane
        ax.scatter(model.ref_scores, ref_y + rng.uniform(-0.03, 0.03, len(model.ref_scores)),
                   s=24, c=TEAL, edgecolors="white", linewidths=0.7, zorder=3)
    lane_y = (ref_y + data_y) / 2 if unsupervised else data_y
    order = sorted(range(len(scores)), key=lambda i: scores[i])
    # Unsupervised piles can hold hundreds of documents, so only anomalies get a name tag there.
    labelled = [i for i in order if verdicts[i] == "anomaly"] if unsupervised else order
    label_rank = {i: r for r, i in enumerate(labelled)}
    for i in order:
        score = scores[i]; col = RED if verdicts[i] == "anomaly" else TEAL
        by_score = score > model.threshold
        flagged = content_flagged[i]
        if flagged and not by_score:
            ax.scatter([score], [lane_y], s=66, facecolors="none", edgecolors=RED, linewidths=1.7, zorder=4)
        else:
            ax.scatter([score], [lane_y], s=46, c=col, edgecolors="white", linewidths=1.1, zorder=4)
            if flagged and by_score:
                ax.scatter([score], [lane_y], s=150, facecolors="none", edgecolors=RED, linewidths=1.3, zorder=4)
        if i not in label_rank:
            continue
        y = levels[label_rank[i] % len(levels)]
        ax.plot([score, score], [lane_y + 0.02, y - 0.03], color=col, lw=0.6, alpha=0.45, zorder=2)
        ax.annotate(_short_name(names[i]), (score, y), ha="center", va="center", fontsize=7, color=col,
                    fontweight="bold" if verdicts[i] == "anomaly" else "normal", zorder=5)
    ax.set_ylim(0, 1.02); ax.set_yticks([]); ax.set_xlim(0, hi)
    ax.spines[["left", "right", "top"]].set_visible(False)
    where = "see the CSV" if unsupervised else "see the grid + CSV"
    ax.text(0.004, 0.02, f"Hollow ring = flagged for a missing required field ({where}), not embedding distance.",
            transform=ax.transAxes, fontsize=7.5, color=GREY, style="italic")
    if unsupervised:
        ax.set_xlabel("Anomaly score — leave-one-out Mahalanobis distance (higher = more unusual vs the rest)", fontsize=9)
        ax.set_ylabel("individual documents (spread vertically to avoid overlap)", fontsize=9)
        ax.set_title("Anomaly scores — unsupervised (each document vs the rest)"
                     + (f" — {title_label}" if title_label else ""),
                     fontsize=12, fontweight="bold", loc="left")
        handles = [
            Line2D([], [], marker="o", ls="", mfc=TEAL, mec="white", label="document — normal"),
            Line2D([], [], marker="o", ls="", mfc=RED, mec="white", label="document — anomaly (past threshold)"),
            Line2D([], [], marker="o", ls="", mfc="none", mec=RED, label="anomaly — missing required field"),
            Line2D([], [], color=GREY, ls="--", label="threshold"),
        ]
    else:
        ax.set_xlabel("Anomaly score — Mahalanobis distance from genuine (higher = more unusual)", fontsize=9)
        ax.set_ylabel("documents (stacked to avoid overlap; vertical position carries no value)", fontsize=9)
        ax.set_title("Distribution of anomaly scores" + (f" — {title_label}" if title_label else ""),
                     fontsize=12, fontweight="bold", loc="left")
        handles = [
            Line2D([], [], marker="o", ls="", mfc=TEAL, mec="white", label="reference (genuine)"),
            Line2D([], [], marker="o", ls="", mfc=TEAL, mec="white", label="data — normal"),
            Line2D([], [], marker="o", ls="", mfc=RED, mec="white", label="anomaly — high score"),
            Line2D([], [], marker="o", ls="", mfc="none", mec=RED, label="anomaly — missing required field"),
            Line2D([], [], color=GREY, ls="--", label="threshold"),
        ]
    ax.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, -0.30),
              ncol=len(handles), fontsize=8, frameon=False)
    fig.savefig(path, dpi=150, bbox_inches="tight"); plt.close(fig)

def plot_score_histogram(scores, threshold, path, title_label=""):
    """The 'distribution' view for unsupervised runs: a histogram of anomaly scores. Bars past the
    threshold are red; the dashed line is the threshold (the PERCENTILE of these same scores)."""
    scores = np.asarray(scores, dtype=float)
    fig, ax = plt.subplots(figsize=(10, 4.6))
    bins = min(45, max(12, int(np.sqrt(len(scores)) * 1.5)))
    counts, edges, patches = ax.hist(scores, bins=bins, color=TEAL, edgecolor="white", linewidth=0.6)
    for patch, left in zip(patches, edges[:-1]):
        if left >= threshold:
            patch.set_facecolor(RED)
    ax.axvline(threshold, color=GREY, ls="--", lw=1.4)
    ax.text(threshold, ax.get_ylim()[1] * 0.98, f"  threshold {threshold:.1f}", color=GREY, fontsize=8, va="top")
    n_anom = int((scores > threshold).sum())
    ax.set_xlabel("Anomaly score — leave-one-out Mahalanobis distance (higher = more unusual)", fontsize=9)
    ax.set_ylabel("number of documents", fontsize=9)
    ax.set_title(f"Distribution of anomaly scores — {n_anom} of {len(scores)} above threshold"
                 + (f" · {title_label}" if title_label else ""), fontsize=12, fontweight="bold", loc="left")
    ax.spines[["right", "top"]].set_visible(False)
    fig.savefig(path, dpi=150, bbox_inches="tight"); plt.close(fig)

def plot_requirement_map(names, details, req_names, path, title_label=""):
    """Grid: one row per document, one column per required field. Green = present (labelled 'exact' or
    the semantic similarity); RED = missing (with the best similarity it reached). The 'which field is
    missing' view."""
    n, m = len(names), len(req_names)
    fig, ax = plt.subplots(figsize=(max(6.0, 1.8 + m * 1.7), max(3.0, 1.2 + n * 0.42)))
    for yi, det in enumerate(details):
        for xi, req in enumerate(req_names):
            method, sim = det.get(req, ("missing", 0.0))
            present = method != "missing"
            ax.add_patch(plt.Rectangle((xi - 0.5, yi - 0.5), 1, 1,
                         facecolor="#dff0ec" if present else "#f6d9d6",
                         edgecolor="#c9d2d0" if present else RED, lw=0.6 if present else 2.2, zorder=1))
            label = "exact" if method == "exact" else (f"sem {sim:.2f}" if method == "semantic" else f"MISS {sim:.2f}")
            ax.text(xi, yi, label, ha="center", va="center", fontsize=7,
                    color=INK if present else RED, fontweight="normal" if present else "bold", zorder=2)
    ax.set_xticks(range(m)); ax.set_xticklabels(req_names, fontsize=8, rotation=15, ha="right")
    ax.set_yticks(range(n)); ax.set_yticklabels([_short_name(x, 26) for x in names], fontsize=8)
    ax.set_xlim(-0.5, m - 0.5); ax.set_ylim(-0.5, n - 0.5); ax.invert_yaxis()
    ax.set_xlabel("Required field", fontsize=9); ax.set_ylabel("Document", fontsize=9)
    ax.set_title("Required-field check — red cell = missing" + (f" · {title_label}" if title_label else ""),
                 fontsize=12, fontweight="bold", loc="left")
    fig.savefig(path, dpi=150, bbox_inches="tight"); plt.close(fig)

## 4 · Run the pipeline

Now we use the code above on your documents. Cells run top-to-bottom.

In [ ]:
# Show matplotlib charts inline, and quieten the informational warnings the model loader prints.
%matplotlib inline
import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

### Run · Stage 1 — load the models, then read text out of a sample document

We build the frozen E5 embedder (first run downloads ~1.1 GB) and the RapidOCR engine, then extract
text from one document so you can see what the model receives. **Every** document is processed the
same way in Stage 2.

When `UNSUPERVISED = True`, `data/` is never read — the pile in `reference/` is scored against itself.

> **Privacy:** the sample text below contains real personal data. It is shown only because this runs
> locally. **Clear this cell's output before sharing the notebook file.**

In [ ]:
embedder = E5TextEmbedder(model_id=MODEL, device=DEVICE)
ocr = RapidOCREngine()
print(f"model : {embedder.model_id}   dim : {embedder.dim}   device : {DEVICE}")

ref_files = list_documents(REFERENCE_DIR)
assert len(ref_files) >= 3, "need at least 3 documents in reference/ to build the model"
if UNSUPERVISED:
    data_files = ref_files          # no ground truth: the pile is scored against itself (leave-one-out)
    print("\nUNSUPERVISED mode: no separate reference — every document is scored vs the REST of the set.")
    print(f"documents in the set: {len(ref_files)}   |   data/ is not read")
else:
    data_files = list_documents(DATA_DIR)
    assert len(data_files) >= 1, "no documents to classify in data/  (or set UNSUPERVISED = True)"
    print(f"reference: {len(ref_files)} documents   |   data: {len(data_files)} documents")

# Extract text from the first document and show it (local demo — clear output before sharing).
sample = ref_files[0]
sample_text = document_to_text(sample.read_bytes(), ocr)
print(f"\nsample document — {sample.name}: {len(sample_text)} characters, {len(_segments(sample_text))} text lines\n")
print("─" * 78)
print(sample_text[:1200] + ("\n… (truncated)" if len(sample_text) > 1200 else ""))
print("─" * 78)

### Run · Stage 2 — text → embedding, then embed every document in `reference/`

`embed_text()` turns a document's text into a 768-number vector. We embed **every** document in
`reference/` into a matrix `X_ref` of shape *(n_documents, 768)* — this is our definition of "normal"
(in unsupervised mode, of "typical for this pile"). We also build one `SemanticMatcher` (reusing the
same E5 model) for the required-field check in Stage 4.

The extracted text is kept in `ref_texts` so that unsupervised runs — where Stage 4 checks these same
documents — do not have to OCR everything a second time.

In [ ]:
sample_vec = np.array(embedder.embed_text(sample_text), dtype=float)
print(f"one embedding -> shape {sample_vec.shape}, first 8 values {np.round(sample_vec[:8], 4)}")

matcher = SemanticMatcher(embedder)

kind = "documents in the set" if UNSUPERVISED else "genuine documents"
print(f"\nembedding {len(ref_files)} {kind} (OCR + E5) — this can take a minute ...")

# Keep each document's text alongside its vector. In UNSUPERVISED mode Stage 4 runs the required-field
# checklist over these very same documents, so caching the text here avoids a second (slow) OCR pass.
# Text is held in memory only — nothing extra is written to disk.
ref_texts, X_ref = [], []
for p in ref_files:
    text = document_to_text(p.read_bytes(), ocr)
    ref_texts.append(text)
    X_ref.append(embedder.embed_text(text))
X_ref = np.array(X_ref, dtype=float)
print(f"reference matrix X_ref: {X_ref.shape}   ({X_ref.shape[0]} {kind})")

### Run · Stage 3 — fit the "normal" model and see where the threshold comes from

`NormalModel.fit` fits the genuine Gaussian and derives the alarm threshold as the **95th percentile
of the genuine leave-one-out scores** (each genuine document scored against all the *others*). The
second cell shows the highest genuine scores and what other percentile choices would give.

In [ ]:
model = NormalModel.fit(X_ref, threshold_percentile=PERCENTILE)
kind = "documents in the set" if UNSUPERVISED else "genuine documents"
print(f"threshold : {model.threshold:.2f}   (= {PERCENTILE:.0f}th percentile of the leave-one-out scores)")
print(f"shrinkage : {model.shrinkage:.2f}   (0 = raw covariance, 1 = fully shrunk to identity)")
print(f"fitted on : {X_ref.shape[0]} {kind}")
print(f"leave-one-out score range: {model.ref_scores.min():.2f} .. {model.ref_scores.max():.2f}")

In [ ]:
loo = model.ref_scores
labels = [p.name.rsplit(".", 1)[0] for p in ref_files]
order = np.argsort(loo)

kind = "documents in the set" if UNSUPERVISED else "genuine documents"
TOP = min(10, len(loo))
print(f"The {TOP} highest-scoring {kind} (these are what set the threshold):\n")
for r, i in enumerate(order[::-1][:TOP], 1):
    print(f"  {r:2}. {_short_name(labels[i], 24):24} {loo[i]:7.2f}  {'#' * int(loo[i] / 2)}")

print("\nThreshold at other percentile choices (what changing PERCENTILE would give):")
for p in (90, 95, 99, 100):
    print(f"  percentile {p:3} -> threshold {np.percentile(loo, p):7.2f}")

### Run · Stage 4 — score every document, apply the required-field rules, decide

Each document is turned into text, embedded and scored, and checked against the required-field rules.
A document is **anomaly** if its score exceeds the threshold **OR** it is missing a required field. We
print the per-document verdict, save a non-identifying CSV (most-unusual first), and draw the charts.

- **Supervised** (`UNSUPERVISED = False`): the `data/` documents are scored against the genuine cloud.
- **Unsupervised** (`UNSUPERVISED = True`): the leave-one-out scores computed during the fit *are* the
  anomaly scores — nothing is scored against itself and no second pass is needed. The field checklist
  still runs, over the text cached in Stage 2.

In [ ]:
import csv
from datetime import datetime

names, scores, verdicts, content_flagged, missing_list, details = [], [], [], [], [], []

if UNSUPERVISED:
    # Honest anomaly score = leave-one-out: each document scored against the OTHERS. Already computed
    # during the fit (model.ref_scores), so there is no separate scoring pass and no self-comparison.
    # The required-field checklist still runs — it needs no reference set — reusing the Stage 2 text.
    print(f"UNSUPERVISED: using the leave-one-out scores of all {len(ref_files)} documents.\n")
    for p, text, sc in zip(ref_files, ref_texts, model.ref_scores):
        content_ok, missing, detail = evaluate_content(text, SPEC, matcher)
        verdict = "anomaly" if (float(sc) > model.threshold or not content_ok) else "normal"
        names.append(p.name); scores.append(float(sc)); verdicts.append(verdict)
        content_flagged.append(not content_ok); missing_list.append(missing); details.append(detail)
else:
    print(f"scoring {len(data_files)} document(s) ...\n")
    for p in data_files:
        text = document_to_text(p.read_bytes(), ocr)      # in-memory only; never written out
        score = model.score(np.array(embedder.embed_text(text), dtype=float))
        content_ok, missing, detail = evaluate_content(text, SPEC, matcher)
        verdict = "anomaly" if (score > model.threshold or not content_ok) else "normal"
        names.append(p.name); scores.append(score); verdicts.append(verdict)
        content_flagged.append(not content_ok); missing_list.append(missing); details.append(detail)

# Per-document table, most unusual first. A large unsupervised pile is truncated — the CSV has it all.
order = sorted(range(len(names)), key=lambda i: -scores[i])
if UNSUPERVISED:
    TOPN = min(20, len(order))
    print(f"Most unusual {TOPN} of {len(names)} documents (highest score first; full list in the CSV):\n")
else:
    TOPN = len(order)
print(f"{'document':26} {'score':>7}  {'content':>7}  {'missing':>22}  verdict")
print("-" * 82)
for i in order[:TOPN]:
    miss = ",".join(missing_list[i]) if missing_list[i] else "-"
    flag = "  <== ANOMALY" if verdicts[i] == "anomaly" else ""
    print(f"{_short_name(names[i],26):26} {scores[i]:7.2f}  {('OK' if not content_flagged[i] else 'FAIL'):>7}  {miss:>22}  {verdicts[i]:8}{flag}")

n_anom = verdicts.count("anomaly")
print(f"\nDOCUMENTS: {n_anom} anomaly, {len(names) - n_anom} normal  (threshold = {model.threshold:.2f})")

# Non-identifying results CSV (scores + field names only — safe to keep/share), most-unusual first.
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_csv = RESULT_DIR / f"{ts}_results.csv"
with out_csv.open("w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["filename", "mahalanobis_score", "content_ok", "missing_requirements", "verdict"])
    for i in order:
        w.writerow([names[i], f"{scores[i]:.2f}", "no" if content_flagged[i] else "yes",
                    "|".join(missing_list[i]), verdicts[i]])
print("saved:", out_csv, " (sorted most-unusual first)")

In [ ]:
import tempfile
from IPython.display import Image as IPyImage, display

req_names = list(SPEC["requirements"])
with tempfile.TemporaryDirectory() as td:
    dist_png = Path(td) / "dist.png"
    plot_distribution(model, names, scores, verdicts, content_flagged, dist_png, DOC_TYPE,
                      unsupervised=UNSUPERVISED)
    display(IPyImage(filename=str(dist_png)))
    if UNSUPERVISED:
        # the "distribution" view — a histogram of scores with the threshold marked. The per-document
        # required-field grid is skipped here: a pile can hold hundreds of rows, which makes the grid
        # unreadable. Which field is missing stays in the table above and in the CSV.
        hist_png = Path(td) / "hist.png"
        plot_score_histogram(scores, model.threshold, hist_png, DOC_TYPE)
        display(IPyImage(filename=str(hist_png)))
    else:
        # the per-document required-field grid (rows = documents, columns = required fields)
        map_png = Path(td) / "map.png"
        plot_requirement_map(names, details, req_names, map_png, DOC_TYPE)
        display(IPyImage(filename=str(map_png)))

## 5 · How to read the result, and how to tune it

**The verdict combines two signals** (a document is `anomaly` if *either* fires):
1. **Embedding score** > threshold — the document's text is unusual versus the genuine set.
2. **A required field is missing** — the field checklist failed.

**Reading the charts**
- *Distribution of anomaly scores:* one dot per document on the score axis. Teal dots low on the axis
  are genuine references; the dashed line is the threshold; data dots past it (red) are score-anomalies.
  A **hollow red ring** marks a document flagged for a *missing required field* — it can sit **left** of
  the threshold (its score alone would be "normal"), which is why the ring is drawn.
- *Required-field check grid:* rows = documents, columns = required fields. A **green** cell means the
  field was found (`exact` = literal label; `sem 0.87` = matched by meaning at that similarity). A
  **red** cell means the field is missing (with the best similarity it reached). This is the
  "which field is missing" view.

**Tuning**
- **Sensitivity of the score:** raise `PERCENTILE` (e.g. 99) for fewer score-alarms, lower for more.
- **Strictness of "field present":** raise `semantic_threshold` in `RULES` to demand closer wording
  (fewer semantic matches), lower it to be more lenient. It was calibrated so all genuine references
  pass; E5 similarities have a high baseline, so keep it in the ~0.82–0.88 range.
- **The rules themselves:** edit `RULES[DOC_TYPE]` — add/remove a required field, or add anchor
  phrases so a real wording stops reading as "missing".
- **The other document type:** change `DOC_TYPE` and Run-All again.

**Things to check before trusting a verdict**
- Every document in `reference/` must be genuine — the model treats them all as the definition of
  "normal", so one bad reference widens what counts as normal.
- A whole document scoring high usually means it's a different layout/type than your references — treat
  a single flag as "look here", not proof of fraud.
- **No labelled fakes yet:** the score threshold is calibrated only against genuine documents (it
  controls the false-alarm rate), not against fraud. A validated cutoff needs some known-fake examples.
- **Self-contained vs. the batch service:** to avoid any system install, this notebook reads the PDF
  text layer with `pypdfium2`; the production `deid-detect-anomalies` tool uses Poppler's `pdftotext`.
  The OCR (RapidOCR) and embedding (E5) are identical, but on documents that carry an embedded text
  layer the two readers differ slightly, which can nudge a score and flip a **borderline** verdict.
  Clear-cut documents agree; treat the batch service as the source of record if numbers must match.
- **Privacy:** the Stage 1 sample text and any OCR text are personal data. The results CSV is
  non-identifying, but **Clear All Outputs before sharing this `.ipynb`.**

## 6 · Unsupervised mode (no ground truth)

Set `UNSUPERVISED = True` when you have **no known-genuine reference set** — just a pile of documents
you want to screen. Put them all in `reference/<type>/`, leave `data/<type>/` empty, and Run-All.

**What changes:** every document is scored by its **leave-one-out** distance — how far it sits from the
cloud formed by *all the other* documents — instead of being compared to a separate reference. The
alarm threshold is still the `PERCENTILE` (95th) of that distribution, so the **most unusual ~5%** are
flagged automatically. Nothing is compared to itself, and you never set a threshold by hand.

**What does *not* change: the required-field checklist.** Unlike the sibling image notebook, the
verdict here still combines two signals — a document is `anomaly` if its score is past the threshold
**or** it is missing a required field. The checklist is absolute (a payslip must carry a worker name,
company, period and income), so it needs no reference set and keeps working with no ground truth. A
document can therefore be flagged while sitting *left* of the threshold line; that is the hollow red
ring on the chart.

**What you get:**
- the **anomaly-score distribution** (a strip plot + a histogram) with the threshold line — the shape
  of your data, and where the unusual tail begins;
- a **"most unusual documents"** list (console, top 20) and `..._results.csv` **sorted worst-first** —
  the documents to inspect are at the top, with their missing fields in the `missing_requirements`
  column.

The per-document required-field **grid is not drawn** in this mode: a pile can hold hundreds of rows,
which makes the grid unreadable. The same information is in the console table and the CSV.

**How to read it:** treat a flagged document as *"looks different from the rest — look here"*, not as
proof. If most of the pile is genuine, the top of the distribution is where odd scans, wrong-type
documents, or tampering tend to surface. Because the unusual documents are *inside* the set, they
slightly widen "normal"; for a sharper pass you can flag the top few percent, remove them, and re-run.

> **Speed:** the threshold is derived by refitting the Gaussian once per document (leave-one-out), which
> grows quadratically with the size of the pile. A few hundred documents takes minutes — the OCR pass in
> Stage 2 usually still dominates.